# 数据清洗

In [3]:
import os
import re
import fitz  # PyMuPDF
import pandas as pd

# ================= 配置 =================
ROOT_DIR = "Sepsis_Papers"
OUTPUT_EXCEL = "Sepsis_Structured_Data.xlsx"

# ================= 1. 增强版清洗函数 =================
def clean_medical_text(text):
    if not text: return ""
    # 去除非法控制字符
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    # 处理医学论文中常见的连字符断行 (例如: sep- sis -> sepsis)
    text = re.sub(r'(\w+)-\s*\n(\w+)', r'\1\2', text)
    # 替换各种空白符为普通空格
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ================= 2. 语义段落切分 (针对医学论文) =================
def split_medical_sections(text):
    """
    根据医学论文标准结构切分内容
    """
    sections = {
        "Abstract": r"Abstract|SUMMARY",
        "Introduction": r"Introduction|BACKGROUND",
        "Methods": r"Methods|PATIENTS AND METHODS|Materials",
        "Results": r"Results|FINDINGS",
        "Discussion": r"Discussion",
        "Conclusion": r"Conclusion|Concluding Remarks"
    }
    
    extracted_data = {}
    remaining_text = text
    
    # 按照章节顺序逆向查找，防止前面的章节包含后面章节的关键词
    keys = list(sections.keys())
    for i in range(len(keys)-1, -1, -1):
        key = keys[i]
        pattern = sections[key]
        # 查找最后一个匹配项
        match = list(re.finditer(pattern, remaining_text, re.IGNORECASE))
        if match:
            last_match = match[-1]
            extracted_data[key] = remaining_text[last_match.end():].strip()
            remaining_text = remaining_text[:last_match.start()]
    
    # 如果完全没匹配到，存入 FullText
    if not extracted_data:
        extracted_data["FullText_Unstructured"] = text
        
    return extracted_data

# ================= 3. 智能 PDF 提取 (处理双栏) =================
def get_smart_text(file_path):
    doc = fitz.open(file_path)
    full_text = ""
    for page in doc:
        # 使用 blocks 模式并排序，能够显著提升双栏阅读顺序的准确性
        blocks = page.get_text("blocks")
        # 按从上到下、从左到右排序 (x0, y0, x1, y1)
        blocks.sort(key=lambda b: (b[1], b[0])) 
        for b in blocks:
            full_text += b[4] + " " 
    doc.close()
    return full_text

# ================= 4. 主程序 =================
final_rows = []

if not os.path.exists(ROOT_DIR):
    print(f"❌ 错误：找不到文件夹 {ROOT_DIR}")
else:
    for root, dirs, files in os.walk(ROOT_DIR):
        category = os.path.basename(root)
        for filename in [f for f in files if f.lower().endswith('.pdf')]:
            file_path = os.path.join(root, filename)
            try:
                # 1. 智能读取
                raw_text = get_smart_text(file_path)
                # 2. 基础清洗
                cleaned_text = clean_medical_text(raw_text)
                # 3. 章节切分
                section_dict = split_medical_sections(cleaned_text)
                
                # 4. 展开存入列表：一篇论文对应多行，方便后续大模型处理
                for sec_name, sec_content in section_dict.items():
                    if len(sec_content) > 20: # 过滤掉太短的无效分段
                        final_rows.append({
                            "Paper_Category": category,
                            "Filename": filename,
                            "Section": sec_name,
                            "Content": sec_content[:10000], # 限制单格长度防止Excel崩溃
                            "Word_Count": len(sec_content)
                        })
            except Exception as e:
                print(f"   ❌ 处理失败 {filename}: {e}")

    # 保存结果
    if final_rows:
        df = pd.DataFrame(final_rows)
        df.to_excel(OUTPUT_EXCEL, index=False)
        print(f"处理完成！数据已按章节切分并保存至: {OUTPUT_EXCEL}")

处理完成！数据已按章节切分并保存至: Sepsis_Structured_Data.xlsx


In [4]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/luran/Desktop/T3/TER/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
MODEL_ID = "google/gemma-2b-it"
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.float16 if device != "cpu" else torch.float32, 
    device_map=device
)

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.56s/it]


In [6]:
df_paper = pd.read_excel("Sepsis_Structured_Data.xlsx")

In [7]:
abstract_text = df_paper[(df_paper['Filename'] == '38084084.pdf') & 
                         (df_paper['Section'] == 'Abstract')]['Content'].values[0]

metadata_prompt = f"""
[Context]
{abstract_text}

[Task]
Read the medical abstract above and extract 3 points. 
Respond ONLY in this JSON format:

{{
  "paper_type": "identify study design here",
  "patient_population": "who are the patients",
  "main_goal": "what is the study trying to find"
}}

Note: Be concise. No explanation.
"""

# 执行生成 (保持参数不变)
chat = [{"role": "user", "content": metadata_prompt}]
input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("🚀 正在执行轻量化第一步：元数据提取...")
with torch.no_grad():
    # 限制生成的长度，防止模型话痨
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1)

res1 = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Step 1 Output ---")
print(res1)

🚀 正在执行轻量化第一步：元数据提取...


/Users/luran/Desktop/T3/TER/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(



--- Step 1 Output ---
{
  "paper_type": "identify study design here",
  "patient_population": "Children with sepsis",
  "main_goal": "To identify evidence-based practices for diagnosis and initial management of children with sepsis"
}


In [10]:
import gc 
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

In [12]:
 # 拼接 Introduction 和 Conclusion 章节
claims_context = df_paper[(df_paper['Filename'] == '38084084.pdf') & 
                          (df_paper['Section'].isin(['Introduction']))]['Content'].str.cat(sep=" ")

# 注意：我们将复杂的要求简化为：核心结论、支持观点、批评观点
claims_prompt = f"""
[Text to Analyze]
{claims_context[:3000]}

[Task]
Identify the main medical arguments. 
Answer ONLY using this exact JSON template:

{{
  "core_claim": "The Phoenix criteria/Sepsis-3 is better than SIRS because...",
  "sub_claims": [
    "Lactate is useful for...",
    "Organ dysfunction predicts..."
  ],
  "criticism": "The old SIRS criteria is bad because..."
}}

Constraint: Do not say 'I cannot'. If unsure, summarize the last paragraph.
"""

# 执行生成 (将 max_new_tokens 稍微调小，强制其直接进入 JSON)
chat = [{"role": "user", "content": claims_prompt}]
input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1)

res2 = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Step 2 Output ---")
print(res2)


--- Step 2 Output ---
{
  "core_claim": "The Phoenix criteria/Sepsis-3 is better than SIRS because it is more accurate in identifying children with sepsis.",
  "sub_claims": [
    "The Phoenix criteria are more sensitive in identifying children with sepsis compared to the SIRS criteria.",
    "The Phoenix criteria are more specific in identifying children with severe sepsis compared to the SIRS criteria."
  ],
  "criticism": "The old SIRS criteria were based on the assumption that sepsis is a systemic inflammatory response, but recent evidence suggests that sepsis is a complex condition involving both pro- and anti-inflammatory mediators."
}


In [13]:
import gc
import re 
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

# 2. 准备文本：锁定 Methods 和 Results，这是证据的产地
# 限制长度在 1500 字符以内，防止缓冲区溢出
evidence_context = df_paper[(df_paper['Filename'] == '38084084.pdf') & 
                            (df_paper['Section'].isin(['Methods', 'Results']))]['Content'].str.cat(sep=" ")

# 清洗干扰字符和超长引用
clean_evidence_text = re.sub(r'\[.*?\]', '', evidence_context)[:3000]

# 3. 设计轻量化 Prompt
# 我们把 Step 2 得到的 res2 传给它，让它找证据
evidence_prompt = f"""
[Claims to verify]
{res2}

[Search Text (Results/Methods)]
{clean_evidence_text}

[Task]
Find specific data (P-values, % mortality, or trial names) that SUPPORT the claims.
Respond ONLY in this JSON format:
{{
  "supporting_data": "list numerical results or trial names found",
  "evidence_strength": "Strong or Weak",
  "data_source": "which trial or database (e.g., SSC, SHIPSS)"
}}
"""

# 4. 执行生成
chat = [{"role": "user", "content": evidence_prompt}]
input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("🚀 正在执行第三步：支持证据提取...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=200, 
        temperature=0.1,
        do_sample=False
    )

res3 = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Step 3 Output ---")
print(res3)

🚀 正在执行第三步：支持证据提取...

--- Step 3 Output ---
{
  "supporting_data": [
    "The Phoenix criteria are more sensitive in identifying children with sepsis compared to the SIRS criteria.",
    "The SHIPPS trial is currently enrolling 8,800 children with suspected septic shock to test the comparative effectiveness of initial resuscitation with balanced/buffered fluids compared to 0.9% saline.",
  ],
  "evidence_strength": "Strong",
  "data_source": "SHIPPS trial"
}


In [15]:
import gc
import re
 
gc.collect()
if device == "mps":
    torch.mps.empty_cache()
 
against_context = df_paper[(df_paper['Filename'] == '38084084.pdf') & 
                           (df_paper['Section'].isin(['Results', 'Conclusion']))]['Content'].str.cat(sep=" ")
 
sentences = re.split(r'\. ', against_context)
filtered_against_text = ". ".join([s for s in sentences if any(k in s.lower() for k in ["however", "limit", "not", "lack", "challenge"])])[:1500]

against_prompt = f"""
[Target Claim]
{res2}

[Search Text]
{filtered_against_text}

[Task]
Identify evidence that WEAKENS or LIMITS the claim. 
Respond ONLY in this JSON format:
{{
  "against_evidence": "describe the limitation or counter-data found",
  "type": "Limitation / Counter-Data / Scope Restriction",
  "impact": "Direct Refutation / Uncertainty / Scope Restriction"
}}
"""
 
chat = [{"role": "user", "content": against_prompt}]
input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("🚀 正在执行第四步：反对/局限证据提取...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=250, 
        temperature=0.1,
        do_sample=False
    )

res4 = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Step 4 Output ---")
print(res4)

🚀 正在执行第四步：反对/局限证据提取...

--- Step 4 Output ---
{
  "against_evidence": "The passage does not provide any evidence to support the claim that the Phoenix criteria are less accurate than the SIRS criteria.",
  "type": "Limitation / Counter-Data / Scope Restriction",
  "impact": "Direct Refutation"
}


In [16]:
import gc

# 1. 显存清理
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

# 2. 汇总前四步的上下文 (只取核心内容以防超限) 
summary_context = f"""
[Metadata]: {res1}
[Claims]: {res2}
[Supports]: {res3}
[Against]: {res4}
"""

# 3. 设计轻量化 RDF 转换 Prompt
# 强制模型输出 Subject -> Predicate -> Object 的三元组 
rdf_prompt = f"""
[Input Data]
{summary_context}

[Task]
Construct a logical RDF chain. 
Respond ONLY with a list of triples in this format: [Subject] --[Relationship]--> [Object]

[Rules]
1. Map Core Claim to Sub-claims.
2. Map Claims to Supporting Data (from Supports).
3. Map Claims to Limitations (from Against).
4. Identify if the logic is 'Complete' or 'Incomplete'.

Example Output:
[Sepsis Definition] --[supported_by]--> [Lactate > 2mmol/L]
[Lactate Evidence] --[strength]--> [Strong]
[Core Claim] --[status]--> [Incomplete: missing conflict resolution]
"""

# 4. 执行生成
chat = [{"role": "user", "content": rdf_prompt}]
input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

print("🚀 正在执行第五步：构建 RDF 逻辑链接图...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, 
        max_new_tokens=400, 
        temperature=0.1,
        do_sample=False
    )

res5 = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Step 5 Final RDF Logic Map ---")
print(res5)

🚀 正在执行第五步：构建 RDF 逻辑链接图...

--- Step 5 Final RDF Logic Map ---
**Triple 1**
Subject: Core Claim
Relationship: --
Object: The Phoenix criteria

**Triple 2**
Subject: Sub-claims
Relationship: --
Object: The Phoenix criteria are more sensitive in identifying children with sepsis compared to the SIRS criteria.

**Triple 3**
Subject: Sub-claims
Relationship: --
Object: The Phoenix criteria are more specific in identifying children with severe sepsis compared to the SIRS criteria.

**Triple 4**
Subject: Claim
Relationship: --
Object: The Phoenix criteria are more accurate in identifying children with sepsis compared to the SIRS criteria.

**Conclusion:**
The logic is "Incomplete". The passage does not provide any evidence to support the claim that the Phoenix criteria are less accurate than the SIRS criteria.


In [17]:
import re

def generate_rdf_turtle(paper_id, res5_output):
    """
    将 Step 5 的输出转化为标准的 RDF Turtle 格式
    """
    # 定义 RDF 命名空间前缀
    turtle_header = f"""@prefix : <http://medical-knowledge.org/sepsis/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

# 论文主体节点
:Paper_{paper_id.replace('.pdf', '')} rdf:type :SepsisResearch .

"""
    
    # 提取三元组的正则表达式 (匹配 [Subject] --[Relationship]--> [Object])
    # 注意：这里会过滤掉 Gemma 输出中可能的干扰字符
    triples = re.findall(r'\[(.*?)\]\s*--\[(.*?)\]-->\s*\[(.*?)\]', res5_output)
    
    turtle_body = ""
    for sub, pred, obj in triples:
        # 清理空格并格式化为合法标识符
        s = sub.strip().replace(" ", "_").replace(">", "gt").replace("<", "lt")
        p = pred.strip().replace(" ", "_")
        o = obj.strip().replace(" ", "_").replace(">", "gt").replace("<", "lt")
        
        # 构建 RDF 语句
        line = f":{s} :{p} :{o} .\n"
        turtle_body += line

    return turtle_header + turtle_body

# ---------------------------------------------------------
# 执行转换
# ---------------------------------------------------------
# 假设 res5 是你上一步生成的字符串结果
# paper_id = "38084084.pdf"

rdf_turtle_code = generate_rdf_turtle("38084084", res5)

print("--- 最终 RDF Turtle 序列 ---")
print(rdf_turtle_code)

# 将结果保存为文件，方便导入图谱软件
with open("sepsis_logic_map.ttl", "w", encoding="utf-8") as f:
    f.write(rdf_turtle_code)
    print("\n✅ RDF 序列已成功保存至: sepsis_logic_map.ttl")

--- 最终 RDF Turtle 序列 ---
@prefix : <http://medical-knowledge.org/sepsis/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

# 论文主体节点
:Paper_38084084 rdf:type :SepsisResearch .



✅ RDF 序列已成功保存至: sepsis_logic_map.ttl


In [5]:
import os
import re
import fitz  # PyMuPDF
import pandas as pd

# ================= 配置 =================
ROOT_DIR = "Sepsis_Papers"
EXTRACT_CHARS = 5000  # 截取长度

# ================= 1. 清洗函数 (修复版) =================
def clean_text_thoroughly(text):
    """
    彻底清洗文本：
    1. 去除 \n, \t, \xa0 等空白符
    2. 【新增】暴力去除 Excel 不支持的非法控制字符 (如 \x02, \x00 等)
    3. 合并多余空格
    """
    if not text: return ""
    
    # 1. 基础替换：把换行、制表符、不换行空格都变成普通空格
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ').replace(u'\xa0', ' ')
    
    # 2. 【核心修复】去除所有不可见的 ASCII 控制字符 (0-31 和 127)
    # Excel 极其讨厌这些隐藏字符
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)
    
    # 3. 合并连续空格
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# ================= 2. 定位与截取函数 =================
def locate_and_cut(text, window_size=12000):
    """
    尝试定位 'Abstract'，如果找不到则从头开始。
    截取指定长度的字符。
    """
    # 忽略大小写搜索 "Abstract" 或 "Conclusion" 或 "Introduction"
    match = re.search(r'(abstract|conclusion|introduction)', text, re.IGNORECASE)
    
    if match:
        start_idx = match.start()
        status = f"Found '{match.group()}' at idx {start_idx}"
        # 从关键词处往后截取
        content = text[start_idx : start_idx + window_size]
    else:
        start_idx = 0
        status = "Keyword not found, reading from start"
        content = text[:window_size]
        
    return content, status

# ================= 3. 主程序 =================
data_list = []

if not os.path.exists(ROOT_DIR):
    print(f"❌ 错误：找不到文件夹 {ROOT_DIR}")
else:
    print(f"🚀 开始扫描并清洗 {ROOT_DIR} 下的文件...\n")

    for root, dirs, files in os.walk(ROOT_DIR):
        category = os.path.basename(root)
        pdf_files = [f for f in files if f.lower().endswith('.pdf')]
        
        if not pdf_files: continue
        
        print(f"📂 处理分类: {category} (共 {len(pdf_files)} 篇)")
        
        for filename in pdf_files:
            file_path = os.path.join(root, filename)
            
            try:
                # 1. 打开 PDF
                doc = fitz.open(file_path)
                raw_text = ""
                for page in doc:
                    raw_text += page.get_text()
                doc.close()
                
                # 2. 彻底清洗
                cleaned_text = clean_text_thoroughly(raw_text)
                
                # 3. 定位截取
                final_content, extract_status = locate_and_cut(cleaned_text, EXTRACT_CHARS)
                
                # 4. 存入列表
                data_list.append({
                    "category": category,
                    "filename": filename,
                    "status": extract_status,
                    "char_count": len(final_content),
                    "content_preview": final_content 
                })
                
            except Exception as e:
                print(f"   ❌ 读取失败 {filename}: {e}")

    # ================= 4. 结果展示 =================
    print("\n" + "="*50)
    print("✅ 数据预处理完成！")
    
    if data_list:
        df = pd.DataFrame(data_list)
        
        # 这次应该不会报错了
        try:
            output_file = "Sepsis_Cleaned_Data_Preview.xlsx"
            df.to_excel(output_file, index=False)
            print(f"💾 已保存预览文件: {output_file}")
        except Exception as e:
            print(f"⚠️ 保存 Excel 依然失败: {e}")
            print("👉 尝试保存为 CSV 作为备选...")
            df.to_csv("Sepsis_Cleaned_Data_Preview.csv", index=False)
            print("💾 已保存为 CSV")

        # 预览
        pd.set_option('display.max_colwidth', 100)
        display(df.head())
        
        print(f"\n📊 统计: 共处理 {len(df)} 篇文档，平均长度 {int(df['char_count'].mean())} 字符。")
    else:
        print("⚠️ 未找到任何有效数据。")

🚀 开始扫描并清洗 Sepsis_Papers 下的文件...

📂 处理分类: Treatment (共 13 篇)
📂 处理分类: Diagnosis (共 12 篇)

✅ 数据预处理完成！
💾 已保存预览文件: Sepsis_Cleaned_Data_Preview.xlsx


,category,filename,status,char_count,content_preview
0,Treatment,39302568.pdf,Found 'Introduction' at idx 1971,5000,Introduction Sepsis is defined as life-threatening organ dysfunction due to a dysregulated host ...
1,Treatment,31747556.pdf,Found 'Abstract' at idx 521,5000,"Abstract Objectives: To present current evidence on the etiology, risk factors, diagnosis, and m..."
2,Treatment,34340174.pdf,Found 'Abstract' at idx 208,5000,Abstract Pleural sepsis stems from an infection within the pleural space typically from an under...
3,Treatment,33236524.pdf,Found 'Introduction' at idx 1656,5000,Introduction Sepsis is deﬁned as “life-threatening organ dysfunction caused by a dysregulated ho...
4,Treatment,38577320.pdf,Found 'Abstract' at idx 830,5000,"Abstract Though there have been developments in clinical care and management, early and accurate..."



📊 统计: 共处理 25 篇文档，平均长度 5000 字符。


# 输入Gemma，设计prompt

In [8]:
import pandas as pd
import torch
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# ================= 1. 准备数据 =================
# 尝试获取之前步骤清洗好的数据
try:
    if 'data_list' in locals() and data_list:
        print("✅ 检测到内存中的 data_list，直接使用...")
        df_clean = pd.DataFrame(data_list)
    else:
        print("⚠️ 内存中未找到 data_list，尝试读取 Excel 文件...")
        df_clean = pd.read_excel("Sepsis_Cleaned_Data_Preview.xlsx")
        print("✅ 成功读取 Excel 文件！")
except Exception as e:
    print(f"❌ 错误：找不到数据源。请确保你运行了上一步的数据清洗代码。\n{e}")
    # 停止运行，防止报错
    raise e

# ================= 2. 加载模型 (离线模式) =================
MODEL_ID = "google/gemma-2b-it"
print(f"⏳ 正在加载模型: {MODEL_ID} ...")
device = "mps" if torch.backends.mps.is_available() else "cpu"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        torch_dtype=torch.float16,
        device_map=device,
        local_files_only=True 
    )
    print("✅ 模型加载就绪！")
except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    raise e

# ================= 3. 核心 Prompt 设计 (Argument Mining) =================

def extract_json_content(text):
    """鲁棒的 JSON 提取器"""
    try: return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try: return json.loads(match.group())
            except: pass
    return None

def analyze_evidence_chain(text_snippet):
    """
    专门用于挖掘【证据链】的 Prompt。
    输入：清洗后的论文片段
    输出：Premise -> Claim 的结构化数据
    """
    # 截取前 5000 字符，确保包含 Abstract 和 Introduction 的核心论述
    input_text = text_snippet[:5000]
    
    chat = [
        { "role": "user", "content": f"""
You are an expert in Argument Mining for medical research. 
Identify the core **Logical Chain** in the following text.

Text:
{input_text}

---
**Mission**: 
Extract the central argument by identifying the factual **EVIDENCE** and the derived **CONCLUSION**.

**Output Format (Strict JSON)**:
{{
    "topic": "Specific medical issue (e.g., Early Sepsis Diagnosis)",
    "evidence": "The raw data, observation, or statistical finding (The 'Because...')",
    "claim": "The inference or proposal derived from the evidence (The 'Therefore...')",
    "relation_type": "Does this paper SUPPORT a new method or CONTRADICT an old one? (Value: 'Support' or 'Contradict')",
    "chain_summary": "Synthesized logic: Because [Evidence], we conclude [Claim]."
}}

Constraint: Output ONLY the JSON string.
""" }
    ]
    
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=500, do_sample=False)
        
    output_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return extract_json_content(output_text), output_text

# ================= 4. 批量执行逻辑挖掘 =================

logic_results = []
print("-" * 60)
print(f"🚀 开始构建证据链 (共 {len(df_clean)} 篇)...")

for index, row in df_clean.iterrows():
    filename = row['filename']
    category = row['category']
    content = row['content_preview'] # 直接使用之前清洗好的内容
    
    # 跳过过短的内容
    if not isinstance(content, str) or len(content) < 500:
        continue
        
    print(f"   ⛓️  挖掘中: {filename} ... ", end="")
    
    # 调用 AI
    chain_data, raw_out = analyze_evidence_chain(content)
    
    if chain_data:
        # 加上文件名和分类，构成完整的知识节点
        chain_data['filename'] = filename
        chain_data['category'] = category
        logic_results.append(chain_data)
        print("✅ 逻辑提取成功")
    else:
        print("⚠️ 结构解析失败")
        # 记录失败以便人工查看
        logic_results.append({
            'filename': filename,
            'category': category,
            'chain_summary': 'PARSE_ERROR',
            'raw_output': raw_out
        })

# ================= 5. 保存并展示知识网络 =================

if logic_results:
    df_logic = pd.DataFrame(logic_results)
    
    # 调整列顺序，使其符合人类阅读逻辑：文件 -> 话题 -> 证据 -> 结论
    cols = ['category', 'filename', 'topic', 'evidence', 'claim', 'relation_type', 'chain_summary']
    final_cols = [c for c in cols if c in df_logic.columns]
    df_logic = df_logic[final_cols]
    
    output_file = "Sepsis_Evidence_Chains.xlsx"
    df_logic.to_excel(output_file, index=False)
    
    print(f"\n🎉 证据链构建完成！已保存为: {output_file}")
    
    # === 关键：展示前几条逻辑链，看看效果 ===
    print("\n🧐 逻辑链预览 (Knowledge Network Preview):")
    for i, row in df_logic.head(3).iterrows():
        print(f"\n📄 {row['filename']} ({row.get('relation_type', 'N/A')}):")
        print(f"   🔍 证据 (Evidence): {row.get('evidence', 'N/A')}")
        print(f"   💡 结论 (Claim):    {row.get('claim', 'N/A')}")
else:
    print("\n⚠️ 未提取到任何逻辑链。")

✅ 检测到内存中的 data_list，直接使用...
⏳ 正在加载模型: google/gemma-2b-it ...


Loading checkpoint shards: 100%|██████████| 2/2 [00:19<00:00,  9.79s/it]


✅ 模型加载就绪！
------------------------------------------------------------
🚀 开始构建证据链 (共 25 篇)...
   ⛓️  挖掘中: 39302568.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 31747556.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 34340174.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 33236524.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 38577320.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 30915218.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 40685448.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 29626019.pdf ... ⚠️ 结构解析失败
   ⛓️  挖掘中: 34446092.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 29022217.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 38084084.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 25834828.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 32978579.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 39676169.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 37904710.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 32699026.pdf ... ⚠️ 结构解析失败
   ⛓️  挖掘中: 36304809.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 30915218.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 29626019.pdf ... ⚠️ 结构解析失败
   ⛓️  挖掘中: 34612847.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 28963294.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 38084084.pdf ... ✅ 逻辑提取成功
   ⛓️  挖掘中: 26903336.pdf ... ⚠️ 结构解析失败
   ⛓️  挖掘中: 39256242.pdf ... 

In [10]:
%pip install rdflib

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 12.8 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rdflib]2m1/2 [rdflib]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import urllib.parse
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS
from rdflib.namespace import DC, FOAF

# ================= 1. 读取数据 =================
# 优先读取内存中的 logic_results，如果没有则读取文件
try:
    if 'logic_results' in locals() and logic_results:
        print("✅ 使用内存中的 logic_results 数据...")
        df = pd.DataFrame(logic_results)
    else:
        file_path = "Sepsis_Evidence_Chains.xlsx"
        print(f"📂 读取文件: {file_path} ...")
        df = pd.read_excel(file_path)
except Exception as e:
    print(f"❌ 无法加载数据: {e}")
    # 这里为了演示，创建一个假数据，防止代码报错无法运行
    df = pd.DataFrame([{
        "filename": "Paper_Demo.pdf", 
        "category": "Diagnosis",
        "topic": "Sepsis Biomarkers",
        "evidence": "PCT levels increased in 90% of septic patients.",
        "claim": "PCT is a valid biomarker.",
        "relation_type": "Support"
    }])

# ================= 2. 初始化 RDF 图谱 =================
g = Graph()

# 定义命名空间 (Namespace)
EX = Namespace("http://example.org/sepsis/") # 自定义前缀
g.bind("ex", EX)
g.bind("dc", DC) # Dublin Core，用于通用元数据

print("🚀 开始构建 RDF 图谱...")

def clean_uri(text):
    """将文本转换为合法的 URI 片段 (去除空格和特殊字符)"""
    if not isinstance(text, str): return "Unknown"
    # 简单的清洗：空格变下划线，URL 编码
    safe_text = urllib.parse.quote(text.strip().replace(" ", "_")[:50]) # 限制长度防止 URI 过长
    return safe_text

# ================= 3. 遍历并构建三元组 =================
for index, row in df.iterrows():
    # 获取原始数据
    filename = str(row.get('filename', 'Unknown_Paper'))
    topic = str(row.get('topic', 'General_Sepsis'))
    evidence_text = str(row.get('evidence', 'No Evidence'))
    claim_text = str(row.get('claim', 'No Claim'))
    relation = str(row.get('relation_type', 'Support')).lower() # 归一化为小写
    
    # --- 创建节点 URI ---
    
    # 1. 论文节点 (Paper Node)
    paper_uri = URIRef(EX[f"Paper/{clean_uri(filename)}"])
    g.add((paper_uri, RDF.type, EX.Paper))
    g.add((paper_uri, DC.title, Literal(filename)))
    
    # 2. 主题节点 (Topic Node) - 这是连接不同论文的枢纽
    topic_uri = URIRef(EX[f"Topic/{clean_uri(topic)}"])
    g.add((topic_uri, RDF.type, EX.Topic))
    g.add((topic_uri, RDFS.label, Literal(topic)))
    
    # 建立连接: Paper --discusses--> Topic
    g.add((paper_uri, EX.discusses, topic_uri))
    
    # 3. 证据节点 (Evidence Node) - 使用哈希或随机ID，这里用 BNode (匿名节点) 也可以，但为了可视化用 URI
    # 这里我们用文本的前30个字符作为 ID
    ev_id = clean_uri(evidence_text[:30])
    evidence_uri = URIRef(EX[f"Evidence/{ev_id}"])
    g.add((evidence_uri, RDF.type, EX.Evidence))
    g.add((evidence_uri, EX.hasText, Literal(evidence_text)))
    
    # 建立连接: Paper --provides--> Evidence
    g.add((paper_uri, EX.providesEvidence, evidence_uri))
    
    # 4. 结论节点 (Claim Node)
    cl_id = clean_uri(claim_text[:30])
    claim_uri = URIRef(EX[f"Claim/{cl_id}"])
    g.add((claim_uri, RDF.type, EX.Claim))
    g.add((claim_uri, EX.hasText, Literal(claim_text)))
    
    # 5. 关键的一步：根据 relation_type 决定边的类型
    # Evidence --[supports/contradicts]--> Claim
    if "contradict" in relation:
        g.add((evidence_uri, EX.contradicts, claim_uri))
        print(f"   ⚡ 发现冲突链条: {filename}")
    else:
        # 默认为 Support
        g.add((evidence_uri, EX.supports, claim_uri))

# ================= 4. 保存与导出 =================
output_rdf = "sepsis_knowledge_graph.ttl"
g.serialize(destination=output_rdf, format="turtle")

print("-" * 60)
print(f"🎉 RDF 图谱构建完成！包含 {len(g)} 个三元组。")
print(f"💾 文件已保存为: {output_rdf}")
print("   (你可以用 Protege, Gephi 或文本编辑器打开这个 .ttl 文件)")

# ================= 5. (可选) 简单的可视化预览 =================
# 仅为了在 Notebook 里快速看一眼结构
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    
    print("\n🔍 生成网络预览图 (仅展示前 50 个节点)...")
    
    # 将 RDF 转换为 NetworkX 图
    nx_graph = nx.DiGraph()
    
    count = 0
    for s, p, o in g:
        # 只把 URI 转换成简单的字符串显示
        s_label = s.split("/")[-1] if isinstance(s, URIRef) else str(s)
        o_label = o.split("/")[-1] if isinstance(o, URIRef) else str(o)
        
        # 简化关系显示
        if "supports" in str(p): edge_label = "supports"
        elif "contradicts" in str(p): edge_label = "contradicts"
        elif "discusses" in str(p): edge_label = "topic"
        else: edge_label = "link"
        
        nx_graph.add_edge(s_label, o_label, relation=edge_label)
        
        count += 1
        if count > 50: break # 防止图太大卡死
            
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(nx_graph, k=0.5)
    nx.draw(nx_graph, pos, with_labels=True, node_size=1500, node_color="skyblue", font_size=8, font_weight="bold", arrowsize=15)
    edge_labels = nx.get_edge_attributes(nx_graph, 'relation')
    nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels)
    plt.title("Sepsis Logic Knowledge Graph (Preview)")
    plt.show()

except ImportError:
    print("⚠️ 缺少 networkx 或 matplotlib，跳过可视化。")

✅ 使用内存中的 logic_results 数据...
🚀 开始构建 RDF 图谱...
   ⚡ 发现冲突链条: 31747556.pdf
   ⚡ 发现冲突链条: 33236524.pdf
   ⚡ 发现冲突链条: 38577320.pdf
   ⚡ 发现冲突链条: 39676169.pdf
   ⚡ 发现冲突链条: 34612847.pdf
------------------------------------------------------------
🎉 RDF 图谱构建完成！包含 188 个三元组。
💾 文件已保存为: sepsis_knowledge_graph.ttl
   (你可以用 Protege, Gephi 或文本编辑器打开这个 .ttl 文件)
⚠️ 缺少 networkx 或 matplotlib，跳过可视化。


In [13]:
%pip install pyvis

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyvis]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install matplotlib

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.3 kB)
Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_universal2.whl (2.9 MB)
Using cached kiwisolver-1.4.9-cp311-cp311-macosx_11_0_arm64.whl (65 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packa

In [20]:
import pandas as pd
import networkx as nx
from pyvis.network import Network
import re

# ================= 1. 数据准备 =================
# 读取之前生成的 Excel (包含 category, evidence, claim, relation_type)
if 'logic_results' in locals() and logic_results:
    df = pd.DataFrame(logic_results)
else:
    try:
        df = pd.read_excel("Sepsis_Evidence_Chains.xlsx")
    except:
        print("❌ 错误：找不到数据源。请确保逻辑链提取步骤已完成。")
        raise

# 过滤掉无效数据
df = df.dropna(subset=['evidence', 'claim'])
print(f"✅ 原始数据: {len(df)} 条逻辑链")

# ================= 2. 语义归一化工具 (简易版) =================
def extract_concept_keyword(text, stop_words=["the", "a", "an", "of", "in", "with", "for", "to", "is", "are", "we", "found"]):
    """
    从长句中提取核心关键词作为节点 ID，以便合并相似观点。
    例如: "High levels of Procalcitonin (PCT)" -> "Procalcitonin (PCT)"
    """
    if not isinstance(text, str): return "Unknown"
    
    # 1. 清洗
    text = re.sub(r'[^\w\s]', '', text) # 去标点
    words = text.split()
    
    # 2. 提取首字母大写的词 (专有名词) 或 核心名词
    # 这里用一个简单策略：取前 3-4 个非停用词作为“短标签”
    meaningful_words = [w for w in words if w.lower() not in stop_words]
    
    if not meaningful_words: return text[:15]
    
    # 取前4个词作为简短 ID (足以区分不同证据，又能在不同论文间撞车)
    short_id = " ".join(meaningful_words[:4]) 
    return short_id

# ================= 3. 构建语义网络 =================
nx_graph = nx.DiGraph()

print("🔄 正在构建交叉重叠网络...")

for index, row in df.iterrows():
    # 1.以此为基础分类 (Diagnosis / Treatment)
    topic_category = row['category'] # 文件夹名字
    
    evidence_raw = row['evidence']
    claim_raw = row['claim']
    relation = row.get('relation_type', 'Support')
    filename = row['filename']
    
    # 2. 生成节点 ID (关键词)
    # 这里的关键是：如果两篇论文提取出的 evidence 关键词一样，它们就会共用同一个节点！
    ev_id = extract_concept_keyword(evidence_raw)
    cl_id = extract_concept_keyword(claim_raw)
    
    # 3. 添加 Evidence 节点
    if ev_id not in nx_graph.nodes:
        nx_graph.add_node(ev_id, 
                          label=ev_id, 
                          title=f"【完整证据】\n{evidence_raw}", # 鼠标悬浮显示完整内容
                          group=topic_category, # 按 Topic 聚类
                          shape='ellipse')
    
    # 4. 添加 Claim 节点
    if cl_id not in nx_graph.nodes:
        nx_graph.add_node(cl_id, 
                          label=cl_id, 
                          title=f"【完整结论】\n{claim_raw}", # 鼠标悬浮显示完整内容
                          group=topic_category,
                          shape='box')
    
    # 5. 添加连线 (Edge)
    # 将“论文来源”藏在边的属性里，不单独作为节点
    edge_color = "#ff4444" if "contradict" in str(relation).lower() else "#4caf50" # 红 vs 绿
    width = 3 if "contradict" in str(relation).lower() else 1
    
    # 如果边已经存在（说明另一篇论文也发现了同样的逻辑），则叠加来源
    if nx_graph.has_edge(ev_id, cl_id):
        old_title = nx_graph.edges[ev_id, cl_id].get('title', '')
        new_title = old_title + f"\nAlso in: {filename}"
        nx_graph.edges[ev_id, cl_id]['title'] = new_title
        # 增加权重，让这条共识边变粗
        nx_graph.edges[ev_id, cl_id]['width'] += 1
    else:
        nx_graph.add_edge(ev_id, cl_id, 
                          title=f"Source: {filename}\nRelation: {relation}", 
                          color=edge_color, 
                          width=width)

# ================= 4. 可视化 (PyVis) =================
print("🎨 正在绘制 Topic 聚类图...")

net = Network(notebook=True, height="900px", width="100%", bgcolor="#1a1a1a", font_color="white", cdn_resources='remote')
net.from_nx(nx_graph)

# --- 物理引擎配置 ---
# 让 Diagnosis 和 Treatment 分开一些，但内部紧凑
net.set_options("""
var options = {
  "nodes": {
    "font": { "size": 16, "color": "white" },
    "borderWidth": 2
  },
  "edges": {
    "color": { "inherit": false },
    "smooth": { "type": "continuous" },
    "arrows": { "to": { "enabled": true, "scaleFactor": 0.5 } }
  },
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.005,
      "springLength": 150,
      "springConstant": 0.05
    },
    "maxVelocity": 40,
    "solver": "forceAtlas2Based"
  },
  "groups": {
    "Diagnosis": { "color": "#2196f3", "shape": "dot" }, 
    "Treatment": { "color": "#ff9800", "shape": "dot" }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100
  }
}
""")

output_html = "sepsis_semantic_network.html"
net.show(output_html)

print(f"\n🎉 语义网络已生成！")
print(f"👉 打开文件: {output_html}")
print("💡 说明:")
print("   1. 蓝色节点 = Diagnosis 阵营，橙色节点 = Treatment 阵营。")
print("   2. 节点上只显示【关键词】(Concept)。")
print("   3. 把鼠标放在节点上，会显示提取出的【完整句子】。")
print("   4. 把鼠标放在连线上，会显示【这条逻辑来自哪篇论文】。")
print("   5. 如果一条线很粗，说明多篇论文都支持这个逻辑！")

✅ 原始数据: 19 条逻辑链
🔄 正在构建交叉重叠网络...
🎨 正在绘制 Topic 聚类图...
sepsis_semantic_network.html

🎉 语义网络已生成！
👉 打开文件: sepsis_semantic_network.html
💡 说明:
   1. 蓝色节点 = Diagnosis 阵营，橙色节点 = Treatment 阵营。
   2. 节点上只显示【关键词】(Concept)。
   3. 把鼠标放在节点上，会显示提取出的【完整句子】。
   4. 把鼠标放在连线上，会显示【这条逻辑来自哪篇论文】。
   5. 如果一条线很粗，说明多篇论文都支持这个逻辑！


In [21]:
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ================= 1. 准备工作 =================
# 加载之前生成的单点证据链
try:
    df_chains = pd.read_excel("Sepsis_Evidence_Chains.xlsx")
    print(f"✅ 已加载 {len(df_chains)} 条证据链。")
except:
    print("❌ 找不到文件，请确保上一步已完成。")
    raise

# 加载模型 (离线模式)
MODEL_ID = "google/gemma-2b-it"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"⏳ Loading Gemma-2b on {device.upper()}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map=device, local_files_only=True
)

# ================= 2. 预处理：按 Topic 分组 =================
# 为了让 Gemma 能处理，我们需要把相似话题的论文聚在一起
# 简单归一化 Topic (转小写，取前两个词作为大致分类)
df_chains['topic_group'] = df_chains['topic'].apply(lambda x: " ".join(str(x).lower().split()[:2]))

# 只保留那些至少有 2 篇论文参与的话题 (否则没法做“跨文档”分析)
topic_counts = df_chains['topic_group'].value_counts()
valid_topics = topic_counts[topic_counts > 1].index
df_filtered = df_chains[df_chains['topic_group'].isin(valid_topics)]

print(f"🔍 将对 {len(valid_topics)} 个话题组进行跨文档分析...")

# ================= 3. 核心推理函数 =================

def extract_relationships_from_llm(topic, records):
    """
    让 LLM 阅读同一话题下的多条 Claim，找出它们的关系。
    """
    # 1. 构建输入文本 (Context)
    # 给每一条 Claim 编一个 ID (如 [P1], [P2]) 方便 LLM 引用
    context_text = ""
    id_map = {} # { "P1": "filename.pdf" }
    
    for idx, row in enumerate(records):
        pid = f"P{idx+1}"
        id_map[pid] = row['filename']
        # 格式: [P1] (Paper_Name): Evidence -> Claim
        context_text += f"[{pid}] ({row['filename']}): {row['chain_summary']}\n"
    
    # 2. 编写 Prompt (这是关键！)
    # 要求 LLM 输出结构化的关系：Source_ID | Relation | Target_ID
    prompt = f"""
You are a senior medical researcher conducting a meta-analysis.
Topic: {topic}

Here are claims extracted from different papers:
{context_text}

**Task**: Identify logical connections BETWEEN these papers.
Does one paper **SUPPORT**, **CONTRADICT**, or **EXTEND** the findings of another?

**Output Format**:
Output a list of relationships. Use exactly this format:
[Source_ID] | [RELATION] | [Target_ID]

Allowed RELATIONS: "SUPPORTS", "CONTRADICTS", "EXTENDS"

Example:
P1 | SUPPORTS | P2
P3 | CONTRADICTS | P1

**Constraint**: Output ONLY the relationship lines. If no direct connection, output "NONE".
"""
    
    # 3. 调用 Gemma
    chat = [{ "role": "user", "content": prompt }]
    inputs = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(inputs, return_tensors="pt").to(device).input_ids
    
    with torch.no_grad():
        outputs = model.generate(input_ids, max_new_tokens=200, do_sample=False)
        
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    
    return response, id_map

# ================= 4. 批量执行 =================

cross_doc_relations = []

for topic in tqdm(valid_topics, desc="Analyzing Topics"):
    # 获取该话题下的所有记录
    group = df_filtered[df_filtered['topic_group'] == topic]
    records = group.to_dict('records')
    
    # 调用 LLM
    raw_response, id_map = extract_relationships_from_llm(topic, records)
    
    # 解析 LLM 的输出
    # 寻找类似 "P1 | SUPPORTS | P2" 的模式
    lines = raw_response.strip().split('\n')
    for line in lines:
        if "|" in line:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) == 3:
                src_pid, rel, tgt_pid = parts
                
                # 将 P1, P2 还原回文件名
                if src_pid in id_map and tgt_pid in id_map:
                    # 找到对应的 Claim 内容用于可视化
                    src_row = group.iloc[int(src_pid[1:])-1]
                    tgt_row = group.iloc[int(tgt_pid[1:])-1]
                    
                    cross_doc_relations.append({
                        "topic": topic,
                        "source_paper": id_map[src_pid],
                        "source_claim": src_row['claim'], # 存下观点用于显示
                        "relation": rel.upper(),
                        "target_paper": id_map[tgt_pid],
                        "target_claim": tgt_row['claim']
                    })

# ================= 5. 保存结果 =================
if cross_doc_relations:
    df_relations = pd.DataFrame(cross_doc_relations)
    output_file = "Sepsis_Cross_Paper_Logic.xlsx"
    df_relations.to_excel(output_file, index=False)
    print(f"\n🎉 跨文档推理完成！发现 {len(df_relations)} 组逻辑联系。")
    print(f"💾 结果已保存: {output_file}")
    print(df_relations.head())
else:
    print("⚠️ 未发现跨文档联系。可能话题太分散，或者模型比较保守。")

✅ 已加载 25 条证据链。
⏳ Loading Gemma-2b on MPS...


Loading checkpoint shards: 100%|██████████| 2/2 [00:23<00:00, 11.85s/it]


🔍 将对 2 个话题组进行跨文档分析...


Analyzing Topics: 100%|██████████| 2/2 [03:11<00:00, 95.60s/it] 

⚠️ 未发现跨文档联系。可能话题太分散，或者模型比较保守。
